In [ ]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

os.chdir('..')
assert os.path.basename(os.getcwd()) == "munich-accident-analysis", \
    "Need to be in the repo directory, restart the notebook and run this cell again"

## Splitting Test Set

In [ ]:
# Set max columns and rows to display
pd.set_option('display.max_columns', 500)
pd.set_option('display.max_rows', 500)

In [ ]:
unfall_table_path = "data/D_Unfall_RData.csv"
unfalltext_table_path = "data/D_Unfalltext_image.csv"
translated_unfalltext_table_path = "data/translated_unfalltext_0_103111.csv"
old_project_test_set_path = "data/old_consulting_project_2023_classification_results.csv"
duplicates_to_remove_path = '/home/accidents/data/contributions_wise2425/duplicated_text_description_list.xlsx'

In [ ]:
unfall_table = pd.read_csv(unfall_table_path)
unfalltext_table = pd.read_csv(unfalltext_table_path)
translated_unfalltext_table = pd.read_csv(translated_unfalltext_table_path)
old_project_test_set = pd.read_csv(old_project_test_set_path)
duplicates_to_remove = pd.read_excel(duplicates_to_remove_path)

In [ ]:
unfall_table.shape

In [ ]:
# Merge all the dataframes
merged_df = unfall_table.merge(unfalltext_table, on="UN_KEY", how="left").merge(
    translated_unfalltext_table[['UN_KEY', 'Translated_Text']], on="UN_KEY", how="left")

In [ ]:
merged_df.shape

In [ ]:
# Remove row if Text is empty
merged_df = merged_df[merged_df['Text'].notnull()]
# Remove duplicates to remove if Text matches
merged_df = merged_df[~merged_df['Text'].isin(duplicates_to_remove['Text'])]
# Remove old project test set
merged_df = merged_df[~merged_df['UN_KEY'].isin(old_project_test_set['UN_KEY'])]
merged_df.shape

In [ ]:
merged_df.head(3)

In [ ]:
# Randomly sample 500 rows
sampled_df = merged_df.sample(500, random_state=42).reset_index(drop=True)
sampled_df = sampled_df[['UN_KEY', 'Text']]
sampled_df.shape

In [ ]:
sampled_df

In [ ]:
sampled_df.to_csv("data/new_test_set_500_samples.csv", index=False, encoding='utf-8')

## Reading Expert Labels

In [ ]:
unfall_table_path = "data/D_Unfall_RData.csv"
unfalltext_table_path = "data/D_Unfalltext_image.csv"
## Old project expert labels
old_project_test_set_path = "data/old_consulting_project_2023_classification_results.csv"
## Expert label
path_to_new_test_set = "data/new_test_set_500_samples_with_expert_labels.xlsx"

In [ ]:
unfall_table = pd.read_csv(unfall_table_path)
unfalltext_table = pd.read_csv(unfalltext_table_path)
old_project_test_set = pd.read_csv(old_project_test_set_path)
new_test_set = pd.read_excel(path_to_new_test_set, engine='openpyxl')

In [ ]:
unfall_table.shape

In [ ]:
# Standardize column names
old_project_test_set.rename(columns={'True_types_SF': 'old_test_set_expert_label'}, inplace=True)
new_test_set.rename(columns={'Typ': 'new_test_set_expert_label'}, inplace=True)

In [ ]:
# Drop duplicates from old project test set
old_project_test_set = old_project_test_set.drop_duplicates(subset=['UN_KEY'])

In [ ]:
# Count numb of expert labels that are not NA
print("Old Project Expert Labels:", old_project_test_set['old_test_set_expert_label'].notnull().sum())
print("New Project Expert Labels: ", new_test_set['new_test_set_expert_label'].notnull().sum())

In [ ]:
# Merge all the dataframes
merged_df = unfall_table.merge(unfalltext_table, on="UN_KEY", how="left").merge(
    old_project_test_set[['UN_KEY', 'old_test_set_expert_label']], on="UN_KEY", how="left").merge(
    new_test_set[['UN_KEY', 'new_test_set_expert_label']], on="UN_KEY", how="left")

In [ ]:
merged_df.shape

In [ ]:
merged_df.head(3)

In [ ]:
# Fuse the expert labels
merged_df['expert_label'] = merged_df['new_test_set_expert_label'].fillna(merged_df['old_test_set_expert_label'])

In [ ]:
print("Expert labels available: ", merged_df['expert_label'].notnull().sum())
print("Expert label distr: ", merged_df.expert_label.value_counts(dropna=False))